# Imports

In [1]:
import pickle

import numpy as np
import pandas as pd

from tqdm import tqdm

from sklearn.model_selection import StratifiedKFold, cross_val_predict

## Utils

In [2]:
def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)

# Loading Datasets

In [3]:
X_train = pd.read_parquet('../data/X_train_stacking_layer_one.parquet')
y_train = pd.read_parquet('../data/y_train.parquet')

X_test = pd.read_parquet('../data/X_test_stacking_layer_one.parquet')

In [4]:
X_train.head()

,lgbm_0,lgbm_1,lgbm_2,cat_0,cat_1,cat_2,xgb_0,xgb_1,xgb_2
0,0.999946,0.000047,6.235225e-06,0.999971,0.000025,3.499764e-06,0.999863,0.000131,0.000006
1,0.988228,0.000484,1.128788e-02,0.986465,0.000105,1.343005e-02,0.988845,0.000337,0.010818
2,0.000006,0.999994,4.529885e-07,0.000006,0.999994,3.879918e-10,0.000007,0.999992,0.000001
3,0.999831,0.000159,9.526890e-06,0.999900,0.000099,3.615600e-07,0.999711,0.000280,0.000009
4,0.998633,0.001335,3.222033e-05,0.999030,0.000962,8.227142e-06,0.999295,0.000690,0.000015


In [5]:
X_test.head()

,lgbm_0,lgbm_1,lgbm_2,cat_0,cat_1,cat_2,xgb_0,xgb_1,xgb_2
0,0.998938,0.000945,0.000117,0.997728,1.893188e-03,3.784761e-04,0.998359,0.001546,0.000096
1,0.997533,0.002457,0.000010,0.997947,2.053297e-03,1.286922e-07,0.998815,0.001179,0.000007
2,0.996906,0.000477,0.002618,0.999850,1.264084e-07,1.494327e-04,0.997993,0.000357,0.001650
3,0.001207,0.000409,0.998383,0.000800,1.348342e-04,9.990656e-01,0.001168,0.000338,0.998494
4,0.999827,0.000158,0.000015,0.999818,1.822315e-04,1.314618e-07,0.999859,0.000132,0.000009


# Machine Learning

In [7]:
models = dict(
    lgbm=load_pickle('../models/layer_2/model_lightgbm.pkl'),
    cat=load_pickle('../models/layer_2/model_catboost.pkl'),
    xgb=load_pickle('../models/layer_2/model_xgboost.pkl'),
    lg=load_pickle('../models/layer_2/model_logistic_regression.pkl'),
    sgd=load_pickle('../models/layer_2/model_sgdclassifier.pkl'),
)

## Train Dataset

In [8]:
cv = StratifiedKFold(shuffle=True, random_state=42, n_splits=5)

In [9]:
X_train_stacking = pd.DataFrame({})

In [10]:
for model_name, model in tqdm(models.items()):
    
    print(f"Predicting Train Dataset {model_name}")

    predictions = cross_val_predict(model, X_train, y_train.class_encoded, cv=cv, n_jobs=-1, method='predict_proba')
    X_train_stacking[[f'{model_name}_0', f'{model_name}_1', f'{model_name}_2']] = predictions

  0%|          | 0/5 [00:00<?, ?it/s]

Predicting Train Dataset lgbm


 20%|██        | 1/5 [03:07<12:28, 187.11s/it]

Predicting Train Dataset cat


 40%|████      | 2/5 [18:02<30:11, 603.75s/it]

...

 60%|██████    | 3/5 [19:26<12:12, 366.39s/it]

Predicting Train Dataset lg


 80%|████████  | 4/5 [19:30<03:43, 223.24s/it]

Predicting Train Dataset sgd


100%|██████████| 5/5 [19:33<00:00, 234.65s/it]


## Test Dataset

In [11]:
X_test_stacking = pd.DataFrame({})

In [12]:
for model_name, model in tqdm(models.items()):
    
    print(f"Predicting Test Dataset {model_name}")
    
    X_test_stacking[[f'{model_name}_0', f'{model_name}_1', f'{model_name}_2']] = model.predict_proba(X_test)

  0%|          | 0/5 [00:00<?, ?it/s]

Predicting Test Dataset lgbm


 20%|██        | 1/5 [00:35<02:22, 35.62s/it]

Predicting Test Dataset cat


 40%|████      | 2/5 [00:36<00:44, 14.90s/it]

Predicting Test Dataset xgb


100%|██████████| 5/5 [00:36<00:00,  7.33s/it]

Predicting Test Dataset lg
Predicting Test Dataset sgd


# Saving

In [13]:
X_train_stacking.to_parquet('../data/X_train_stacking_layer_two.parquet')
X_test_stacking.to_parquet('../data/X_test_stacking_layer_two.parquet')

In [14]:
X_train_stacking.head()

,lgbm_0,lgbm_1,lgbm_2,cat_0,cat_1,cat_2,xgb_0,xgb_1,xgb_2,lg_0,lg_1,lg_2,sgd_0,sgd_1,sgd_2
0,0.999910,0.000083,0.000007,0.999866,0.000124,0.000010,0.999928,0.000070,0.000003,0.988210,0.004367,0.007423,0.999210,0.000790,0.000000
1,0.992961,0.000411,0.006628,0.991422,0.000567,0.008011,0.992562,0.000566,0.006872,0.986847,0.004779,0.008373,0.988522,0.001245,0.010233
2,0.000109,0.999858,0.000033,0.000162,0.999819,0.000019,0.000030,0.999962,0.000008,0.009897,0.988895,0.001208,0.002604,0.997396,0.000000
3,0.999805,0.000187,0.000009,0.999766,0.000223,0.000010,0.999837,0.000159,0.000004,0.988202,0.004372,0.007426,0.999098,0.000902,0.000000
4,0.998393,0.001560,0.000047,0.998544,0.001413,0.000043,0.998238,0.001711,0.000051,0.987966,0.004405,0.007629,0.998207,0.001793,0.000000


In [15]:
X_test_stacking.head()

,lgbm_0,lgbm_1,lgbm_2,cat_0,cat_1,cat_2,xgb_0,xgb_1,xgb_2,lg_0,lg_1,lg_2,sgd_0,sgd_1,sgd_2
0,0.997841,0.002113,0.000045,0.997856,0.001945,0.000199,0.998030,0.001934,0.000036,0.987950,0.004507,0.007543,0.997783,0.002217,0.000000
1,0.996887,0.003080,0.000033,0.997599,0.002381,0.000020,0.996744,0.003240,0.000016,0.987886,0.004549,0.007565,0.997324,0.002676,0.000000
2,0.998244,0.001026,0.000731,0.998669,0.000563,0.000768,0.998349,0.000750,0.000901,0.987787,0.004504,0.007709,0.998949,0.001051,0.000000
3,0.000658,0.000086,0.999256,0.001490,0.000130,0.998380,0.000538,0.000138,0.999324,0.013634,0.004166,0.982199,0.000000,0.002127,0.997873
4,0.999840,0.000151,0.000009,0.999810,0.000178,0.000012,0.999880,0.000116,0.000004,0.988019,0.004456,0.007524,0.999069,0.000931,0.000000
